# 🏠 Arquitetura Medallion - Visão Geral

Este notebook executa o pipeline real do projeto nas 3 camadas, com **data quality** entre elas:

1. **Bronze**: ingestão dos CSVs brutos (`data/`) para tabelas Delta + check
2. **Silver**: limpeza, tipagem e normalização + check
3. **Gold**: agregações de negócio + check

Stack: Spark + Delta Lake + MinIO (**padrão ligado** — para warehouse local use `USE_MINIO=0`).

> **Dados faltando?** Se `data/` estiver vazio (repo clonado sem CSVs): `python -m scripts.generate_data`
>
> **Mesma sequência no Airflow**: DAG `medallion_pipeline` com as tasks `bronze >> check >> silver >> check >> gold >> check` — veja a seção final.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.session import get_spark
from src.ingestion.Bronze import run as run_bronze
from src.processing.Silver import run as run_silver
from src.serving.Gold import run as run_gold
from src.dq import checks as dq

spark = get_spark("MedallionOverview")
print("Spark Session criada com sucesso!")

## 1. Camada BRONZE — Ingestão de Dados Brutos

In [ ]:
run_bronze(spark)
spark.table("bronze.orders").show(5, truncate=False)

### ✔ Data Quality — Bronze
Sanidade estrutural: tabelas existem, não vazias, `_ingested_at` presente.

In [ ]:
report = dq.run_bronze(spark)
report.log_summary()
assert report.ok, f"{len(report.errors)} erro(s) de DQ na Bronze"

## 2. Camada SILVER — Limpeza e Tipagem

In [ ]:
run_silver(spark, preview=True)

### ✔ Data Quality — Silver
PKs únicas, `rating` 1–5, domínio de `status`, referencialidade + **WARNs** de valores negativos (não bloqueiam).

In [ ]:
report = dq.run_silver(spark)
report.log_summary()
assert report.ok, f"{len(report.errors)} erro(s) de DQ na Silver"

## 3. Camada GOLD — Agregações de Negócio

In [ ]:
run_gold(spark, show=True)

### ✔ Data Quality — Gold
Métricas não nulas e `avaliacao_media` entre 1 e 5.

In [ ]:
report = dq.run_gold(spark)
report.log_summary()
assert report.ok, f"{len(report.errors)} erro(s) de DQ na Gold"

## 4. Consultas nas tabelas Gold

In [ ]:
print("=== Vendas por Categoria ===")
spark.table("gold.vendas_por_categoria").show(truncate=False)

print("=== Pedidos por Status ===")
spark.table("gold.pedidos_por_status").show(truncate=False)

In [ ]:
print("=== Top 10 Clientes por Gasto ===")
spark.table("gold.resumo_clientes").limit(10).show(truncate=False)

In [ ]:
print("=== SQL: receita total por status ===")
spark.sql('''
    SELECT status,
           SUM(receita_total) AS receita,
           SUM(total_pedidos) AS pedidos
    FROM gold.pedidos_por_status
    GROUP BY status
    ORDER BY receita DESC
''').show(truncate=False)

## 5. Histórico de Transações Delta

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "bronze.orders")
history = delta_table.history()

print("=== Histórico (bronze.orders) ===")
history.select(
    "version",
    "timestamp",
    "operation",
    "numOutputRows",
).show(truncate=False)

## 6. Orquestração com Airflow

Esta mesma sequência (com os checks entre as camadas) roda automaticamente na DAG **`medallion_pipeline`**:

```
bronze_ingest >> check_bronze >> silver_process >> check_silver >> gold_aggregate >> check_gold
```

```bash
podman-compose up -d --build
# UI: http://localhost:8080 (admin / admin)
podman exec medallion_airflow_scheduler airflow dags trigger medallion_pipeline
```

Agendamento: `@daily`, `catchup=False`, `max_active_runs=1` (Derby metastore não aceita escrita concorrente).

In [ ]:
spark.stop()
print("\nSessão Spark encerrada.")